# Error analysis in Colab (Pascal VOC + YOLOv8n)

Runs the Day 2 error-analysis workflow end to end:

1. Install dependencies.
2. Prepare the VOC2007 test split (`val`) if `data/images/val` is empty.
3. Locate or upload `yolov8n_voc_best.pt`.
4. Rank false positives/negatives, class confusion, and recall by object size.
5. Sweep confidence and NMS IoU thresholds.
6. Write `results/error_analysis.md`, `results/error_analysis.json`, and sweep plots.

Run the cells top to bottom. If the checkpoint is not in the runtime, the
checkpoint cell asks you to upload it.


In [ ]:
%pip install -q ultralytics pandas matplotlib pyyaml

## 1. Project root

Uses the current directory when it already contains the project, otherwise
clones the public repository into `/content/02-Object-Detection`.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Albusxx-01/02-Object-Detection.git"
REPO_DIR = Path("/content/02-Object-Detection")


def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents, REPO_DIR):
        if (candidate / "config" / "detection.yaml").exists():
            return candidate
    return Path.cwd()


ROOT = find_project_root()
if not (ROOT / "config" / "detection.yaml").exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
        check=True,
    )
    ROOT = REPO_DIR

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)


## 2. Configuration

Edit these knobs before running the rest of the notebook:

- `WEIGHTS_PATH`: optional explicit checkpoint path.
- `SWEEP_LIMIT`: `None` runs sweeps on the full validation split; an integer
  caps the sweep to that many images.
- `FULL_SWEEP`: `True` uses 19 confidence values and 7 IoU values.
- `AUTO_LIMIT_ON_CPU`: automatically caps sweeps to 500 images on CPU.


In [ ]:
import json
import shutil
import urllib.request
import zipfile
import xml.etree.ElementTree as ET
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml

ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
RESULTS_DIR = ROOT / "results"
MODELS_DIR = ROOT / "models"
DATASET_YAML = DATA_DIR / "dataset.yaml"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

config = yaml.safe_load(
    (ROOT / "config" / "detection.yaml").read_text(encoding="utf-8")
)
CLASS_NAMES = [
    str(config["names"][index]) for index in sorted(config["names"])
]

SPLIT = "val"
CONF = 0.25
IOU = 0.45
IMGSZ = 640
IOU_THRESHOLD = 0.5
SIZE_BUCKETS = (
    (0.0, 0.01, "tiny"),
    (0.01, 0.05, "small"),
    (0.05, 0.2, "medium"),
    (0.2, float("inf"), "large"),
)
DEVICE = 0 if torch.cuda.is_available() else "cpu"

WEIGHTS_PATH = ""
SWEEP_LIMIT = None
FULL_SWEEP = True
AUTO_LIMIT_ON_CPU = True

print("Classes:", len(CLASS_NAMES))
print("Device:", DEVICE)


## 3. Prepare the validation split

If `data/images/val` already exists (for example from notebook 01), this cell
skips the download. Otherwise it downloads VOC2007 test and converts the XML
annotations to YOLO labels.


In [ ]:
VOC_TEST_URL = (
    "https://github.com/ultralytics/assets/releases/download/v0.0.0/"
    "VOCtest_06-Nov-2007.zip"
)
VOC_TEST_SIZE = 438_316_827


def download(url: str, destination: Path) -> Path:
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and destination.stat().st_size >= VOC_TEST_SIZE:
        return destination
    print("Downloading", destination.name)
    urllib.request.urlretrieve(url, destination)
    return destination


def convert_box(size: tuple[int, int], box: list[float]) -> tuple[float, ...]:
    width, height = size
    x_min, x_max, y_min, y_max = box
    x_center = ((x_min + x_max) / 2.0 - 1.0) / width
    y_center = ((y_min + y_max) / 2.0 - 1.0) / height
    box_width = (x_max - x_min) / width
    box_height = (y_max - y_min) / height
    return x_center, y_center, box_width, box_height


def convert_annotation(xml_path: Path, label_path: Path) -> int:
    root = ET.parse(xml_path).getroot()
    size_node = root.find("size")
    image_size = (
        int(size_node.find("width").text),
        int(size_node.find("height").text),
    )
    lines = []
    for obj in root.iter("object"):
        class_name = obj.find("name").text
        if class_name not in CLASS_NAMES:
            continue
        difficult = obj.find("difficult")
        if difficult is not None and int(difficult.text) == 1:
            continue
        box_node = obj.find("bndbox")
        box = [
            float(box_node.find(key).text)
            for key in ("xmin", "xmax", "ymin", "ymax")
        ]
        coordinates = convert_box(image_size, box)
        coordinates = [min(max(value, 0.0), 1.0) for value in coordinates]
        if coordinates[2] <= 0.0 or coordinates[3] <= 0.0:
            continue
        class_id = CLASS_NAMES.index(class_name)
        values = [str(class_id)] + [f"{value:.6f}" for value in coordinates]
        lines.append(" ".join(values))
    label_path.parent.mkdir(parents=True, exist_ok=True)
    label_path.write_text(
        "\n".join(lines) + ("\n" if lines else ""), encoding="utf-8"
    )
    return len(lines)


images_dir = DATA_DIR / "images" / SPLIT
labels_dir = DATA_DIR / "labels" / SPLIT
image_paths = sorted(images_dir.glob("*.jpg"))

if not image_paths:
    archive = DATA_DIR / "downloads" / "VOCtest_06-Nov-2007.zip"
    download(VOC_TEST_URL, archive)
    devkit = DATA_DIR / "VOCdevkit"
    marker = devkit / ".VOCtest_06-Nov-2007.zip.extracted"
    if not marker.exists():
        print("Extracting", archive.name)
        with zipfile.ZipFile(archive) as archive_file:
            archive_file.extractall(DATA_DIR)
        marker.touch()
    year_dir = devkit / "VOC2007"
    image_ids = (
        year_dir / "ImageSets" / "Main" / "test.txt"
    ).read_text(encoding="utf-8").strip().split()
    images_dir.mkdir(parents=True, exist_ok=True)
    labels_dir.mkdir(parents=True, exist_ok=True)
    count = 0
    for image_id in image_ids:
        source_image = year_dir / "JPEGImages" / f"{image_id}.jpg"
        source_xml = year_dir / "Annotations" / f"{image_id}.xml"
        if not source_image.exists():
            continue
        stem = f"2007_{image_id}"
        shutil.copy2(source_image, images_dir / f"{stem}.jpg")
        destination_label = labels_dir / f"{stem}.txt"
        if source_xml.exists():
            convert_annotation(source_xml, destination_label)
        else:
            destination_label.write_text("", encoding="utf-8")
        count += 1
    image_paths = sorted(images_dir.glob("*.jpg"))
    print("Converted val images:", count)
else:
    print("Using existing val split:", len(image_paths), "images")

DATASET_YAML.write_text(
    yaml.safe_dump(
        {
            "path": str(DATA_DIR.resolve()),
            "train": f"images/{SPLIT}",
            "val": f"images/{SPLIT}",
            "test": f"images/{SPLIT}",
            "names": {index: name for index, name in enumerate(CLASS_NAMES)},
        },
        sort_keys=False,
    ),
    encoding="utf-8",
)
print("Dataset YAML:", DATASET_YAML)


## 4. Locate the trained checkpoint

Set `WEIGHTS_PATH` in the configuration cell if you know the path. Otherwise
this cell searches the usual project locations and asks you to upload
`yolov8n_voc_best.pt` if it is missing.


In [ ]:
candidates = []
if WEIGHTS_PATH:
    candidates.append(Path(WEIGHTS_PATH))
candidates.extend(
    [
        MODELS_DIR / "yolov8n_voc_best.pt",
        RESULTS_DIR / "runs" / "yolov8n_voc" / "weights" / "best.pt",
        Path("/content/models/yolov8n_voc_best.pt"),
        Path("/content/results/runs/yolov8n_voc/weights/best.pt"),
    ]
)
weights = next((path for path in candidates if path.exists()), None)

if weights is None:
    print("Checkpoint not found. Upload models/yolov8n_voc_best.pt.")
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError(
            "Upload yolov8n_voc_best.pt or set WEIGHTS_PATH."
        ) from exc
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError("No checkpoint uploaded.")
    uploaded_path = Path(next(iter(uploaded)))
    weights = MODELS_DIR / uploaded_path.name
    if uploaded_path.resolve() != weights.resolve():
        shutil.move(str(uploaded_path), weights)

print("Checkpoint:", weights)


In [ ]:
from ultralytics import YOLO

model = YOLO(str(weights))
if isinstance(model.names, dict):
    names = [model.names[index] for index in sorted(model.names)]
else:
    names = list(model.names)
print("Model classes:", len(names))


## 5. Match predictions to ground truth

Predictions are matched greedily by descending confidence at IoU >= 0.5.
A prediction that matches an already-used ground-truth box is a false
positive; any unmatched ground-truth box is a false negative.


In [ ]:
def box_iou(boxes_a, boxes_b):
    if len(boxes_a) == 0 or len(boxes_b) == 0:
        return np.zeros((len(boxes_a), len(boxes_b)), dtype=np.float32)
    top_left = np.maximum(boxes_a[:, None, :2], boxes_b[None, :, :2])
    bottom_right = np.minimum(boxes_a[:, None, 2:], boxes_b[None, :, 2:])
    sizes = np.clip(bottom_right - top_left, 0.0, None)
    intersection = sizes[..., 0] * sizes[..., 1]
    area_a = np.clip(boxes_a[:, 2] - boxes_a[:, 0], 0.0, None)
    area_a *= np.clip(boxes_a[:, 3] - boxes_a[:, 1], 0.0, None)
    area_b = np.clip(boxes_b[:, 2] - boxes_b[:, 0], 0.0, None)
    area_b *= np.clip(boxes_b[:, 3] - boxes_b[:, 1], 0.0, None)
    union = area_a[:, None] + area_b[None, :] - intersection
    return np.where(union > 0.0, intersection / union, 0.0).astype(np.float32)


def load_ground_truth(label_path, image_shape):
    if not label_path.exists() or label_path.stat().st_size == 0:
        return np.zeros((0, 4), dtype=np.float32), np.zeros((0,), dtype=int)
    labels = np.loadtxt(label_path, ndmin=2, dtype=np.float32)
    height, width = image_shape
    scale = np.array([width, height, width, height], dtype=np.float32)
    boxes = labels[:, 1:5] * scale
    xyxy = np.column_stack(
        (
            boxes[:, 0] - boxes[:, 2] / 2.0,
            boxes[:, 1] - boxes[:, 3] / 2.0,
            boxes[:, 0] + boxes[:, 2] / 2.0,
            boxes[:, 1] + boxes[:, 3] / 2.0,
        )
    )
    return xyxy.astype(np.float32), labels[:, 0].astype(int)


def match_detections(
    gt_boxes, gt_classes, pred_boxes, pred_scores, pred_classes
):
    ious = box_iou(gt_boxes, pred_boxes)
    matched_gt, true_positives, false_positives = set(), [], []
    for pred_index in np.argsort(-pred_scores):
        if len(gt_boxes) == 0:
            false_positives.append(int(pred_index))
            continue
        candidate = ious[:, pred_index].copy()
        candidate = np.where(
            gt_classes == pred_classes[pred_index], candidate, 0.0
        )
        if matched_gt:
            candidate[list(matched_gt)] = -1.0
        gt_index = int(np.argmax(candidate))
        if candidate[gt_index] >= IOU_THRESHOLD:
            matched_gt.add(gt_index)
            true_positives.append((gt_index, int(pred_index)))
        else:
            false_positives.append(int(pred_index))
    false_negatives = [
        index for index in range(len(gt_boxes)) if index not in matched_gt
    ]
    return {
        "ious": ious,
        "true_positives": true_positives,
        "false_positives": false_positives,
        "false_negatives": false_negatives,
    }


def size_bucket(area):
    for lower, upper, label in SIZE_BUCKETS:
        if lower <= area < upper:
            return label
    return SIZE_BUCKETS[-1][2]


def normalized_area(box, image_shape):
    height, width = image_shape
    if height <= 0 or width <= 0:
        return 0.0
    area = max(box[2] - box[0], 0.0) * max(box[3] - box[1], 0.0)
    return float(area / (width * height))


## 6. Collect false positives and false negatives

Each false negative records the closest prediction, its confidence, and its IoU
so misses can be ranked even when no box was produced.


In [ ]:
false_positives = []
false_negatives = []
confusion = Counter()
buckets = {
    label: {"total": 0, "matched": 0} for _, _, label in SIZE_BUCKETS
}
total_gt = 0
total_predictions = 0
total_true_positives = 0

results = model.predict(
    source=str(images_dir / "*.jpg"),
    conf=CONF,
    iou=IOU,
    imgsz=IMGSZ,
    verbose=False,
    stream=True,
    device=DEVICE,
)
for result in results:
    image_path = Path(result.path)
    image_shape = tuple(int(value) for value in result.orig_shape)
    label_path = labels_dir / f"{image_path.stem}.txt"
    gt_boxes, gt_classes = load_ground_truth(label_path, image_shape)

    boxes = getattr(result, "boxes", None)
    if boxes is None or len(boxes) == 0:
        pred_boxes = np.zeros((0, 4), dtype=np.float32)
        pred_scores = np.zeros((0,), dtype=np.float32)
        pred_classes = np.zeros((0,), dtype=int)
    else:
        pred_boxes = boxes.xyxy.cpu().numpy().astype(np.float32)
        pred_scores = boxes.conf.cpu().numpy().astype(np.float32)
        pred_classes = boxes.cls.cpu().numpy().astype(int)

    match = match_detections(
        gt_boxes, gt_classes, pred_boxes, pred_scores, pred_classes
    )
    ious = match["ious"]

    total_gt += len(gt_boxes)
    total_predictions += len(pred_boxes)
    total_true_positives += len(match["true_positives"])

    for gt_index, _ in match["true_positives"]:
        area = normalized_area(gt_boxes[gt_index], image_shape)
        bucket = size_bucket(area)
        buckets[bucket]["total"] += 1
        buckets[bucket]["matched"] += 1

    for gt_index in match["false_negatives"]:
        area = normalized_area(gt_boxes[gt_index], image_shape)
        buckets[size_bucket(area)]["total"] += 1
        best_confidence = 0.0
        best_iou = 0.0
        best_class = None
        if len(pred_scores) > 0:
            pred_index = int(np.argmax(ious[gt_index]))
            best_iou = float(ious[gt_index, pred_index])
            best_confidence = float(pred_scores[pred_index])
            best_class = names[int(pred_classes[pred_index])]
        class_id = int(gt_classes[gt_index])
        false_negatives.append(
            {
                "image": image_path.name,
                "class_name": names[class_id],
                "confidence": best_confidence,
                "best_iou": best_iou,
                "best_class": best_class,
                "area": area,
            }
        )

    for pred_index in match["false_positives"]:
        class_id = int(pred_classes[pred_index])
        false_positives.append(
            {
                "image": image_path.name,
                "class_name": names[class_id],
                "confidence": float(pred_scores[pred_index]),
                "area": normalized_area(pred_boxes[pred_index], image_shape),
            }
        )

    if len(gt_boxes) > 0 and len(pred_scores) > 0:
        for gt_index, gt_class_id in enumerate(gt_classes):
            pred_index = int(np.argmax(ious[gt_index]))
            if ious[gt_index, pred_index] < IOU_THRESHOLD:
                continue
            actual = names[int(gt_class_id)]
            predicted = names[int(pred_classes[pred_index])]
            confusion[(actual, predicted)] += 1

false_positives.sort(key=lambda item: item["confidence"], reverse=True)
false_negatives.sort(
    key=lambda item: (item["confidence"], -item["area"]), reverse=True
)

precision = (
    total_true_positives / total_predictions if total_predictions else 0.0
)
recall = total_true_positives / total_gt if total_gt else 0.0
summary = pd.Series(
    {
        "images": len(image_paths),
        "ground_truth": total_gt,
        "predictions": total_predictions,
        "true_positives": total_true_positives,
        "false_positives": len(false_positives),
        "false_negatives": len(false_negatives),
        "precision": precision,
        "recall": recall,
    },
    name="value",
)
summary.to_frame()


In [ ]:
display(pd.DataFrame(false_positives[:20]))
display(pd.DataFrame(false_negatives[:20]))


In [ ]:
confusion_rows = [
    {"actual": actual, "predicted": predicted, "count": count}
    for (actual, predicted), count in confusion.most_common()
]
pd.DataFrame(confusion_rows).head(20)


In [ ]:
size_rows = []
for label, counts in buckets.items():
    total = counts["total"]
    matched = counts["matched"]
    size_rows.append(
        {
            "bucket": label,
            "ground_truth": total,
            "matched": matched,
            "recall": matched / total if total else 0.0,
        }
    )
pd.DataFrame(size_rows)


## 7. Confidence and NMS IoU sweeps

Each value runs a full `model.val()`. On CPU the notebook caps the sweep to the
first 500 validation images by default; set `SWEEP_LIMIT = None` and
`AUTO_LIMIT_ON_CPU = False` for a full sweep. Set `FULL_SWEEP = False` for a
smaller 5 confidence + 4 IoU sweep.


In [ ]:
if SWEEP_LIMIT is None and AUTO_LIMIT_ON_CPU and DEVICE == "cpu":
    SWEEP_LIMIT = 500
    print("CPU detected: sweeping the first 500 validation images.")

if FULL_SWEEP:
    CONF_VALUES = [round(0.05 * index, 2) for index in range(1, 20)]
    IOU_VALUES = [round(0.1 * index, 2) for index in range(3, 10)]
else:
    CONF_VALUES = [0.10, 0.25, 0.40, 0.55, 0.70]
    IOU_VALUES = [0.30, 0.45, 0.60, 0.75]

sweep_yaml = DATASET_YAML
SWEEP_NOTE = None
if SWEEP_LIMIT is not None:
    sweep_dir = DATA_DIR / "sweeps"
    subset_images = sweep_dir / "images" / SPLIT
    subset_labels = sweep_dir / "labels" / SPLIT
    for directory in (subset_images, subset_labels):
        if directory.exists():
            shutil.rmtree(directory)
        directory.mkdir(parents=True, exist_ok=True)
    for image_path in image_paths[:SWEEP_LIMIT]:
        shutil.copy2(image_path, subset_images / image_path.name)
        label_path = labels_dir / f"{image_path.stem}.txt"
        if label_path.exists():
            shutil.copy2(label_path, subset_labels / label_path.name)
    sweep_yaml = sweep_dir / "dataset.yaml"
    sweep_yaml.write_text(
        yaml.safe_dump(
            {
                "path": str(sweep_dir.resolve()),
                "train": f"images/{SPLIT}",
                "val": f"images/{SPLIT}",
                "test": f"images/{SPLIT}",
                "names": {
                    index: name for index, name in enumerate(CLASS_NAMES)
                },
            },
            sort_keys=False,
        ),
        encoding="utf-8",
    )
    SWEEP_NOTE = f"first {SWEEP_LIMIT} images from {images_dir}"
    print("Sweep subset:", SWEEP_LIMIT, "images")

METRIC_KEYS = {
    "precision": "metrics/precision(B)",
    "recall": "metrics/recall(B)",
    "mAP50": "metrics/mAP50(B)",
    "mAP50-95": "metrics/mAP50-95(B)",
}


def sweep(parameter, values):
    rows = []
    for value in values:
        kwargs = {
            "data": str(sweep_yaml),
            "imgsz": IMGSZ,
            "batch": 16,
            "plots": False,
            "verbose": False,
            "project": str(RESULTS_DIR / "runs"),
            "name": f"error_sweep_{parameter}_{value:g}",
            "exist_ok": True,
            parameter: value,
            "device": DEVICE,
        }
        metrics = model.val(**kwargs)
        row = {"value": value}
        row.update(
            {
                name: float(metrics.results_dict[key])
                for name, key in METRIC_KEYS.items()
                if key in metrics.results_dict
            }
        )
        rows.append(row)
        print(parameter, value, row)
    return pd.DataFrame(rows)


conf_sweep = sweep("conf", CONF_VALUES)
iou_sweep = sweep("iou", IOU_VALUES)
display(conf_sweep)
display(iou_sweep)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, frame, parameter in (
    (axes[0], conf_sweep, "conf"),
    (axes[1], iou_sweep, "iou"),
):
    for metric in ("precision", "recall", "mAP50", "mAP50-95"):
        if metric in frame.columns:
            ax.plot(frame["value"], frame[metric], marker="o", label=metric)
    ax.set_xlabel(parameter)
    ax.set_ylabel("Score")
    ax.set_title(f"{parameter} threshold sweep")
    ax.grid(alpha=0.3)
    ax.legend()
fig.tight_layout()
fig.savefig(
    RESULTS_DIR / "threshold_sweeps.png", dpi=150, bbox_inches="tight"
)
plt.show()

for frame, parameter in ((conf_sweep, "conf"), (iou_sweep, "iou")):
    fig, ax = plt.subplots(figsize=(8, 4.5))
    for metric in ("precision", "recall", "mAP50", "mAP50-95"):
        if metric in frame.columns:
            ax.plot(frame["value"], frame[metric], marker="o", label=metric)
    ax.set_xlabel(parameter)
    ax.set_ylabel("Score")
    ax.set_title(f"{parameter} threshold sweep")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(
        RESULTS_DIR / f"{parameter}_sweep.png",
        dpi=150,
        bbox_inches="tight",
    )
    plt.close(fig)


In [ ]:
best_conf = float(
    conf_sweep.sort_values("mAP50-95", ascending=False).iloc[0]["value"]
)
best_iou = float(
    iou_sweep.sort_values("mAP50-95", ascending=False).iloc[0]["value"]
)
print("Best confidence:", best_conf)
print("Best IoU:", best_iou)


## 8. Write the report

Writes the same `results/error_analysis.md` and `results/error_analysis.json`
schema as `src/error_analysis.py`.


In [ ]:
size_summary = {}
for label, counts in buckets.items():
    total = counts["total"]
    matched = counts["matched"]
    size_summary[label] = {
        "total": total,
        "matched": matched,
        "recall": matched / total if total else 0.0,
    }

analysis = {
    "weights": str(weights),
    "split": SPLIT,
    "images": len(image_paths),
    "ground_truth": total_gt,
    "predictions": total_predictions,
    "true_positives": total_true_positives,
    "false_positive_count": len(false_positives),
    "false_negative_count": len(false_negatives),
    "precision": precision,
    "recall": recall,
    "false_positives": false_positives,
    "false_negatives": false_negatives,
    "class_confusion": [
        {"actual": actual, "predicted": predicted, "count": count}
        for (actual, predicted), count in confusion.most_common()
    ],
    "size_buckets": size_summary,
}
json_path = RESULTS_DIR / "error_analysis.json"
json_path.write_text(
    json.dumps(analysis, indent=2) + "\n", encoding="utf-8"
)
print("JSON:", json_path)


In [ ]:
def markdown_table(headers, rows):
    lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join("---" for _ in headers) + " |",
    ]
    lines.extend(
        "| " + " | ".join(str(value) for value in row) + " |"
        for row in rows
    )
    return "\n".join(lines)


def score(value):
    return f"{value:.4f}"


def sweep_table(frame, parameter):
    if frame.empty:
        return "Not run."
    best = frame.sort_values("mAP50-95", ascending=False).iloc[0]["value"]
    table = markdown_table(
        [parameter, "precision", "recall", "mAP50", "mAP50-95"],
        [
            [
                f"{row['value']:g}",
                score(row.get("precision", 0.0)),
                score(row.get("recall", 0.0)),
                score(row.get("mAP50", 0.0)),
                score(row.get("mAP50-95", 0.0)),
            ]
            for _, row in frame.iterrows()
        ],
    )
    return f"Best {parameter}: {best:g}\n\n{table}"


lines = [
    "# Error analysis",
    "",
    f"- Weights: `{weights}`",
    f"- Split: `{SPLIT}`",
    f"- Images: {len(image_paths)}",
    *([f"- Sweep subset: {SWEEP_NOTE}"] if SWEEP_NOTE else []),
    "",
    "## Summary",
    "",
    markdown_table(
        ["Metric", "Value"],
        [
            ["Ground-truth boxes", total_gt],
            ["Predictions", total_predictions],
            ["True positives", total_true_positives],
            ["False positives", len(false_positives)],
            ["False negatives", len(false_negatives)],
            ["Precision", score(summary["precision"])],
            ["Recall", score(summary["recall"])],
        ],
    ),
    "",
    "## Top false positives by confidence",
    "",
    (
        markdown_table(
            ["Image", "Class", "Confidence", "Area"],
            [
                [
                    item["image"],
                    item["class_name"],
                    score(item["confidence"]),
                    score(item["area"]),
                ]
                for item in false_positives[:20]
            ],
        )
        if false_positives
        else "None."
    ),
    "",
    "## Top false negatives by confidence",
    "",
    (
        markdown_table(
            [
                "Image",
                "Class",
                "Confidence",
                "Best IoU",
                "Best prediction",
                "Area",
            ],
            [
                [
                    item["image"],
                    item["class_name"],
                    score(item["confidence"]),
                    score(item["best_iou"]),
                    item["best_class"] or "none",
                    score(item["area"]),
                ]
                for item in false_negatives[:20]
            ],
        )
        if false_negatives
        else "None."
    ),
    "",
    "## Class confusion",
    "",
    (
        markdown_table(
            ["Actual", "Predicted", "Count"],
            [
                [row["actual"], row["predicted"], row["count"]]
                for row in confusion_rows[:20]
            ],
        )
        if confusion_rows
        else "None."
    ),
    "",
    "## Recall by object size",
    "",
    markdown_table(
        ["Bucket", "Ground truth", "Matched", "Recall"],
        [
            [
                row["bucket"],
                row["ground_truth"],
                row["matched"],
                score(row["recall"]),
            ]
            for row in size_rows
        ],
    ),
    "",
    "## Confidence sweep",
    "",
    sweep_table(conf_sweep, "conf"),
    "",
    "## IoU sweep",
    "",
    sweep_table(iou_sweep, "iou"),
]
report_path = RESULTS_DIR / "error_analysis.md"
report_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
print("Saved:", report_path)


In [ ]:
from IPython.display import Markdown, display

display(Markdown(report_path.read_text(encoding="utf-8")))


In [ ]:
bundle = RESULTS_DIR / "error_analysis_colab.zip"
with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for name in (
        "error_analysis.md",
        "error_analysis.json",
        "threshold_sweeps.png",
        "conf_sweep.png",
        "iou_sweep.png",
    ):
        path = RESULTS_DIR / name
        if path.exists():
            archive.write(path, arcname=name)
print("Bundle:", bundle)
try:
    from google.colab import files
    files.download(str(bundle))
except ImportError:
    print("Download the bundle manually:", bundle)


## Next steps

- Copy `error_analysis.md` and `error_analysis.json` into the repository's
  `results/` directory.
- Commit them with `docs(results): record error-analysis findings`.
- Use the best confidence/IoU in `src/infer.py` and `app.py`.
